# Kalshi WNBA Trading Backtest — Convergence Exit

Walk-forward protocol: train XGB on 2015-2024 (early-stop on 2024), predict 2025.  
sigma_p = ensemble std dev (20 bootstrap models).

**Backtest mechanics**
- Entry decision: single snapshot at tipoff - 5 minutes
- Executable prices: `yes_ask` to buy YES, `1 - yes_bid` to buy NO
- Exit: sell position when market odds converge to model prediction
  - YES position: exit when `yes_bid >= p_xgb`
  - NO position: exit when `yes_ask <= p_xgb` (i.e. `no_bid >= 1 - p_xgb`)
- Monitoring: every 1 minute after tipoff until convergence or settlement
- Cost: zero explicit fee — spread cost already captured by executing at BBO
- One entry per market per strategy; if both sides pass, take larger net edge

**Policies tested**

| Group | Policy | Criterion |
|---|---|---|
| Edge-only | E1 | \|edge\| >= 0.03 |
| Edge-only | E2 | \|edge\| >= 0.05 |
| Edge-only | E3 | \|edge\| >= 0.08 |
| Edge-only | E4 | \|edge\| >= 0.12 |
| Edge+sigma | S1 | \|edge\| >= 0.05, sigma <= 0.05 |
| Edge+sigma | S2 | \|edge\| >= 0.08, sigma <= 0.05 |
| Edge+sigma | S3 | \|edge\| >= 0.08, sigma <= 0.04 |
| Ratio | Z2 | \|edge\| >= 0.05, z >= 1.5 |

z = |edge| / (sigma_p + 0.01)

In [13]:
import warnings; warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd
import xgboost as xgb

PROJECT_ROOT = Path('..').resolve()
GOLD_DIR     = PROJECT_ROOT / 'data' / 'gold'
KALSHI_DIR   = PROJECT_ROOT / 'data' / 'kalshi'

# --- XGB params: tuning3 winner (confirmed by follow-up grid) ---
XGB_PARAMS = dict(
    objective='binary:logistic', eval_metric='logloss',
    max_depth=6, min_child_weight=2, subsample=0.9,
    colsample_bytree=0.8, reg_lambda=0.5, reg_alpha=0.0,
    gamma=0.5, learning_rate=0.03, seed=42, nthread=-1,
)
N_PLAYERS  = 7
MAX_ROUNDS = 3000
EARLY_STOP = 150
N_ENS      = 20
ENS_SEED   = 42
COST       = 0.0           # spread cost already in BBO execution prices
CLIP_EPS   = 1e-7
LABEL_COL  = 'home_win'

# --- Feature columns (same as 32_run_xgboost_cv.py) ---
PLAYER_FEATS = ['m_ewma_pre','q_pre','days_since_first_report_pre','days_since_last_dnp_pre',
                'consec_dnps_pre','played_last_game_pre','minutes_last_game_pre',
                'days_since_last_played_pre','injury_present_flag_pre']
FORM_FEATS   = ['net_rtg_ewma_pre','efg_ewma_pre','tov_pct_ewma_pre','orb_pct_ewma_pre','ftr_ewma_pre']
STYLE_FEATS  = ['off_3pa_rate_pre','def_3pa_allowed_pre','off_2pa_rate_pre','def_2pa_allowed_pre',
                'off_tov_pct_pre','def_forced_tov_pre']
SCHED_FEATS  = ['days_rest_pre','is_b2b_pre','games_last_4_days_pre','games_last_7_days_pre',
                'travel_miles_pre','timezone_shift_hours_pre']

def build_feature_cols(n):
    cols = [f'{s}_p{k}_{f}' for s in ('home','away') for k in range(1, n+1) for f in PLAYER_FEATS]
    for f in FORM_FEATS + STYLE_FEATS + SCHED_FEATS:
        cols += [f'home_{f}', f'away_{f}']
    return cols

FEAT_COLS = build_feature_cols(N_PLAYERS)

def clip(p): return np.clip(p, CLIP_EPS, 1 - CLIP_EPS)

def load_year(year):
    df = pd.read_csv(GOLD_DIR / f'game_xgboost_input_{year}_REGPST.csv')
    cold = (df['home_p1_m_ewma_pre'] == 0) | (df['away_p1_m_ewma_pre'] == 0)
    return df[~cold].reset_index(drop=True)

def make_dm(df, use_bm=True):
    avail = [c for c in FEAT_COLS if c in df.columns]
    y = df[LABEL_COL].values.astype(float) if LABEL_COL in df.columns else None
    dm = xgb.DMatrix(df[avail].values.astype(float), label=y,
                     feature_names=avail, missing=np.nan)
    if use_bm and 'base_margin' in df.columns:
        dm.set_base_margin(df['base_margin'].values.astype(float))
    return dm

print(f'Feature cols: {len(FEAT_COLS)}')
print('Setup complete.')

Feature cols: 160
Setup complete.


## 2. Train XGB + Ensemble on 2015-2024, predict 2025

Produces `p_xgb` (single model) and `sigma_p` (ensemble std) for every 2025 game.

In [14]:
# Load all gold data
all_data = {}
for yr in range(2015, 2026):
    all_data[yr] = load_year(yr)
    print(f'  {yr}: {len(all_data[yr])} rows')

test_2025    = all_data[2025]
es_tr        = pd.concat([all_data[yr] for yr in range(2015, 2024)], ignore_index=True)
val_df       = all_data[2024]
full_tr      = pd.concat([all_data[yr] for yr in range(2015, 2025)], ignore_index=True)

# --- Step 1: early-stop on 2024 to find best_round ---
m_es = xgb.train(
    XGB_PARAMS, make_dm(es_tr), MAX_ROUNDS,
    evals=[(make_dm(val_df), 'val')],
    early_stopping_rounds=EARLY_STOP, verbose_eval=False,
)
best_round = m_es.best_iteration + 1
print(f'\nES best_round: {best_round}')

# --- Step 2: retrain on 2015-2024, predict 2025 ---
m_final = xgb.train(XGB_PARAMS, make_dm(full_tr), best_round, verbose_eval=False)
p_xgb_25 = clip(m_final.predict(make_dm(test_2025)))

# --- Step 3: ensemble (20 bootstrap models) for sigma_p ---
print(f'Training ensemble: {N_ENS} models, {best_round} rounds each...')
ens_preds = []
for i in range(N_ENS):
    seed = ENS_SEED + i
    rng  = np.random.default_rng(seed)
    idx  = rng.integers(0, len(full_tr), size=len(full_tr))
    dm_b = make_dm(full_tr.iloc[idx].reset_index(drop=True))
    m_b  = xgb.train({**XGB_PARAMS, 'seed': seed}, dm_b, best_round, verbose_eval=False)
    ens_preds.append(clip(m_b.predict(make_dm(test_2025))))

ens_stack  = np.stack(ens_preds)
p_ens_mean = ens_stack.mean(0)
sigma_p    = ens_stack.std(0)

# --- Build game-level signal table ---
signals = test_2025[['game_id', 'game_ts', 'game_date',
                      'home_team_id', 'away_team_id', LABEL_COL]].copy()
signals['p_xgb']   = p_xgb_25
signals['sigma_p']  = sigma_p
signals['game_ts']  = pd.to_datetime(signals['game_ts'], utc=True)

from sklearn.metrics import log_loss
y25 = test_2025[LABEL_COL].values.astype(float)
print(f'\nXGB  2025 log-loss: {log_loss(y25, p_xgb_25):.5f}')
print(f'Ens  2025 log-loss: {log_loss(y25, p_ens_mean):.5f}')
print(f'sigma_p: mean={sigma_p.mean():.4f}  median={np.median(sigma_p):.4f}  max={sigma_p.max():.4f}')
print(f'Games: {len(signals)}')

  2015: 218 rows
  2016: 220 rows
  2017: 219 rows
  2018: 221 rows
  2019: 220 rows
  2020: 147 rows
  2021: 209 rows
  2022: 239 rows
  2023: 260 rows
  2024: 262 rows
  2025: 310 rows

ES best_round: 17
Training ensemble: 20 models, 17 rounds each...

XGB  2025 log-loss: 0.61080
Ens  2025 log-loss: 0.61117
sigma_p: mean=0.0314  median=0.0315  max=0.0505
Games: 310


## 3. Load Kalshi data and build home-team market mapping

Each Kalshi event has two market tickers (one per team). We identify which ticker
corresponds to the home team so we can interpret YES/NO sides correctly.

In [15]:
# --- Load Kalshi matched games ---
k_matched = pd.read_csv(KALSHI_DIR / 'wnba_2025_game_markets_matched.csv')
k_markets = pd.read_csv(
    KALSHI_DIR / 'kalshi_markets.csv',
    usecols=['market_ticker', 'event_ticker', 'yes_sub_title', 'result'],
)
k_settle = pd.read_csv(
    KALSHI_DIR / 'kalshi_settlements.csv',
    usecols=['market_ticker', 'result', 'settlement_value_dollars'],
)

# For each game, find the home-team market ticker
# k_matched has team_a, team_a_id, team_b, team_b_id
# k_markets has yes_sub_title (team name) per ticker
game_market = (
    k_matched[['event_ticker', 'game_id', 'team_a', 'team_a_id', 'team_b', 'team_b_id']]
    .merge(k_markets, on='event_ticker', how='inner')
)

# Join with signals to know which team is home
game_market = game_market.merge(
    signals[['game_id', 'home_team_id', 'away_team_id', 'game_ts', 'p_xgb', 'sigma_p', LABEL_COL]],
    on='game_id', how='inner',
)

# Determine if this ticker is the home-team ticker
# yes_sub_title matches team_a or team_b; team_a_id / team_b_id tells us which
game_market['ticker_team_id'] = np.where(
    game_market['yes_sub_title'] == game_market['team_a'],
    game_market['team_a_id'],
    game_market['team_b_id'],
)
game_market['is_home_ticker'] = game_market['ticker_team_id'] == game_market['home_team_id']

# Keep only home-team tickers (one row per game)
home_tickers = game_market[game_market['is_home_ticker']].copy()
home_tickers = home_tickers.drop_duplicates('game_id')

# Settlement: did home team win?  (result='yes' on home ticker means home won)
home_tickers = home_tickers.merge(
    k_settle[['market_ticker', 'result']].rename(columns={'result': 'settle_result'}),
    on='market_ticker', how='left',
)
home_tickers['home_won_settle'] = (home_tickers['settle_result'] == 'yes').astype(int)

print(f'Games with Kalshi home-team ticker: {len(home_tickers)}')
print(f'Settlement check: home_won_settle matches home_win: '
      f'{(home_tickers["home_won_settle"] == home_tickers[LABEL_COL]).all()}')

Games with Kalshi home-team ticker: 296
Settlement check: home_won_settle matches home_win: True


## 4. Build entry snapshots and post-tipoff candle series

- **Entry**: last candle at or before tipoff - 5 min
- **Exit monitoring**: all 1-min candles after tipoff (used to detect convergence)

In [16]:
# Load ALL 1-min candles for home-team tickers
home_ticker_set = set(home_tickers['market_ticker'])
candles = pd.read_csv(
    KALSHI_DIR / 'kalshi_candles_1m.csv',
    usecols=['market_ticker', 'end_period_ts', 'yes_bid_close', 'yes_ask_close'],
)
candles['ts'] = pd.to_datetime(candles['end_period_ts'], utc=True)
candles = candles[candles['market_ticker'].isin(home_ticker_set)].copy()
candles = candles.sort_values(['market_ticker', 'ts']).reset_index(drop=True)
print(f'Home-team candles loaded: {len(candles):,}')

# Build lookup: ticker -> game info
ticker_info = home_tickers.set_index('market_ticker')[
    ['game_id', 'game_ts', 'p_xgb', 'sigma_p', LABEL_COL]
].to_dict('index')

# --- Entry snapshots: last candle at or before (tipoff - 5 min) ---
entry_rows = []
for ticker, info in ticker_info.items():
    tipoff = info['game_ts']
    entry_cutoff = tipoff - pd.Timedelta(minutes=5)

    sub = candles[(candles['market_ticker'] == ticker) & (candles['ts'] <= entry_cutoff)]
    if sub.empty:
        continue

    last = sub.iloc[-1]  # already sorted by ts
    yb, ya = last['yes_bid_close'], last['yes_ask_close']
    if not (0 < yb < 1 and 0 < ya < 1):
        continue

    entry_rows.append({
        'game_id':       info['game_id'],
        'market_ticker': ticker,
        'entry_ts':      last['ts'],
        'yes_bid_entry': yb,
        'yes_ask_entry': ya,
        'q_yes_exec':    ya,           # cost to buy YES
        'q_no_exec':     1 - yb,       # cost to buy NO
        'p_xgb':         info['p_xgb'],
        'sigma_p':       info['sigma_p'],
        'home_win':      info[LABEL_COL],
        'tipoff':        tipoff,
    })

entries = pd.DataFrame(entry_rows)
print(f'Games with valid entry snapshot: {len(entries)}')
print(f'Entry times: median {(entries["tipoff"] - entries["entry_ts"]).median()} before tipoff')

# --- Post-tipoff candle series (for exit monitoring) ---
# Store as dict: market_ticker -> DataFrame of post-tipoff candles
post_tipoff = {}
for ticker, info in ticker_info.items():
    tipoff = info['game_ts']
    sub = candles[(candles['market_ticker'] == ticker) & (candles['ts'] > tipoff)].copy()
    if not sub.empty:
        post_tipoff[ticker] = sub[['ts', 'yes_bid_close', 'yes_ask_close']].reset_index(drop=True)

print(f'Games with post-tipoff candles: {len(post_tipoff)}')

Home-team candles loaded: 235,349
Games with valid entry snapshot: 296
Entry times: median 0 days 00:05:00 before tipoff
Games with post-tipoff candles: 296


## 5. Trading engine — convergence exit

1. At tipoff - 5 min, check entry criteria for each policy
2. Post-tipoff, scan 1-min candles for convergence:
   - YES position: exit when `yes_bid >= p_xgb`
   - NO position: exit when `yes_ask <= p_xgb`
3. If convergence never occurs, hold to settlement

In [17]:
# --- Policy definitions ---
def make_edge_policy(min_edge):
    def policy(ey, en, sigma):
        return max(ey, en) >= min_edge
    return policy

def make_edge_sigma_policy(min_edge, max_sigma):
    def policy(ey, en, sigma):
        return sigma <= max_sigma and max(ey, en) >= min_edge
    return policy

def make_z_policy(min_edge, min_z):
    def policy(ey, en, sigma):
        best_edge = max(ey, en)
        return best_edge >= min_edge and best_edge / (sigma + 0.01) >= min_z
    return policy

POLICIES = {
    'E1': make_edge_policy(0.03),
    'E2': make_edge_policy(0.05),
    'E3': make_edge_policy(0.08),
    'E4': make_edge_policy(0.12),
    'S1': make_edge_sigma_policy(0.05, 0.05),
    'S2': make_edge_sigma_policy(0.08, 0.05),
    'S3': make_edge_sigma_policy(0.08, 0.04),
    'Z2': make_z_policy(0.05, 1.5),
}

# --- Trading engine with convergence exit ---
def run_backtest(entries_df, post_tipoff_candles, policies, cost=COST):
    all_trades = {name: [] for name in policies}

    for _, row in entries_df.iterrows():
        p       = row['p_xgb']
        sigma   = row['sigma_p']
        q_yes   = row['q_yes_exec']
        q_no    = row['q_no_exec']
        hw      = row['home_win']
        ticker  = row['market_ticker']
        game_id = row['game_id']
        tipoff  = row['tipoff']

        edge_yes = p - q_yes
        edge_no  = (1 - p) - q_no

        for pname, pfunc in policies.items():
            if not pfunc(edge_yes, edge_no, sigma):
                continue

            # Determine side
            if edge_yes >= edge_no:
                side     = 'YES'
                entry_px = q_yes
                edge     = edge_yes
            else:
                side     = 'NO'
                entry_px = q_no
                edge     = edge_no

            # --- Search for convergence exit in post-tipoff candles ---
            exit_type = 'settlement'
            exit_px   = None
            exit_ts   = None
            mins_held = None

            if ticker in post_tipoff_candles:
                pt = post_tipoff_candles[ticker]
                for _, c in pt.iterrows():
                    yb = c['yes_bid_close']
                    ya = c['yes_ask_close']
                    if not (0 < yb < 1 and 0 < ya < 1):
                        continue

                    if side == 'YES' and yb >= p:
                        # Market bid reached our model price — sell YES
                        exit_type = 'converged'
                        exit_px   = yb
                        exit_ts   = c['ts']
                        mins_held = (c['ts'] - tipoff).total_seconds() / 60
                        break
                    elif side == 'NO' and ya <= p:
                        # Market ask dropped to our model price — sell NO
                        exit_type = 'converged'
                        exit_px   = 1 - ya   # NO sell price = 1 - yes_ask
                        exit_ts   = c['ts']
                        mins_held = (c['ts'] - tipoff).total_seconds() / 60
                        break

            # --- Compute PnL ---
            if exit_type == 'converged':
                pnl = exit_px - entry_px - 2 * cost   # cost on entry + exit
            else:
                # Hold to settlement
                payout = 1.0 if (side == 'YES' and hw == 1) or (side == 'NO' and hw == 0) else 0.0
                exit_px = payout
                pnl = payout - entry_px - cost          # cost on entry only

            all_trades[pname].append({
                'game_id':     game_id,
                'side':        side,
                'entry_px':    entry_px,
                'edge':        edge,
                'exit_type':   exit_type,
                'exit_px':     exit_px,
                'pnl':         pnl,
                'p_xgb':       p,
                'sigma_p':     sigma,
                'home_win':    hw,
                'mins_held':   mins_held,
            })

    return {k: pd.DataFrame(v) for k, v in all_trades.items()}


trades = run_backtest(entries, post_tipoff, POLICIES)

for pname in POLICIES:
    tdf = trades[pname]
    if tdf.empty:
        print(f'{pname}:   0 trades')
        continue
    n_conv = (tdf['exit_type'] == 'converged').sum()
    n_sett = (tdf['exit_type'] == 'settlement').sum()
    print(f'{pname}: {len(tdf):3d} trades | {n_conv:3d} converged, {n_sett:3d} settlement')

E1: 223 trades | 164 converged,  59 settlement
E2: 182 trades | 128 converged,  54 settlement
E3: 132 trades |  92 converged,  40 settlement
E4:  77 trades |  51 converged,  26 settlement
S1: 180 trades | 128 converged,  52 settlement
S2: 132 trades |  92 converged,  40 settlement
S3: 105 trades |  73 converged,  32 settlement
Z2: 155 trades | 107 converged,  48 settlement


## 6. Results

In [18]:
from sklearn.metrics import brier_score_loss

summary_rows = []

for pname in POLICIES:
    tdf = trades[pname]
    if tdf.empty:
        summary_rows.append({'policy': pname, 'n_trades': 0})
        continue

    n         = len(tdf)
    n_conv    = (tdf['exit_type'] == 'converged').sum()
    n_sett    = (tdf['exit_type'] == 'settlement').sum()
    hit       = (tdf['pnl'] > 0).mean()
    mean_e    = tdf['edge'].mean()
    mean_pnl  = tdf['pnl'].mean()
    total_pnl = tdf['pnl'].sum()
    total_risk = tdf['entry_px'].sum()
    roi       = total_pnl / total_risk if total_risk > 0 else 0

    # Brier: model prob vs actual outcome on the side we traded
    brier_p = np.where(tdf['side'] == 'YES', tdf['p_xgb'], 1 - tdf['p_xgb'])
    brier_y = np.where(tdf['side'] == 'YES', tdf['home_win'], 1 - tdf['home_win'])
    brier_val = brier_score_loss(brier_y, brier_p)

    avg_sigma = tdf['sigma_p'].mean()

    conv = tdf[tdf['exit_type'] == 'converged']
    avg_mins_held = conv['mins_held'].mean() if len(conv) > 0 else np.nan

    summary_rows.append({
        'policy':       pname,
        'n_trades':     n,
        'n_converged':  n_conv,
        'n_settlement': n_sett,
        'mean_edge':    round(mean_e, 4),
        'hit_rate':     round(hit, 4),
        'mean_pnl':     round(mean_pnl, 4),
        'total_pnl':    round(total_pnl, 2),
        'roi':          round(roi, 4),
        'brier':        round(brier_val, 5),
        'avg_sigma_p':  round(avg_sigma, 4),
        'avg_mins_held': round(avg_mins_held, 1) if not np.isnan(avg_mins_held) else None,
    })

summary = pd.DataFrame(summary_rows)

pd.set_option('display.width', 180)
pd.set_option('display.float_format', '{:.4f}'.format)
print('=' * 140)
print('TRADING BACKTEST — Convergence Exit  (2025 season, zero explicit fee, spread in BBO)')
print('=' * 140)
print(summary.to_string(index=False))

# Save
out_dir = PROJECT_ROOT / 'data' / 'kalshi'
summary.to_csv(out_dir / 'backtest_convergence_summary.csv', index=False)
print(f'\nSaved: {out_dir}/backtest_convergence_summary.csv')

# --- Per-policy detail ---
print('\n' + '=' * 140)
for pname in POLICIES:
    tdf = trades[pname]
    if tdf.empty:
        continue
    conv = tdf[tdf['exit_type'] == 'converged']
    sett = tdf[tdf['exit_type'] == 'settlement']
    print(f'\n--- {pname}: {len(tdf)} trades | '
          f'converged={len(conv)} (PnL={conv["pnl"].sum():.2f}) | '
          f'settlement={len(sett)} (PnL={sett["pnl"].sum():.2f}) | '
          f'total PnL={tdf["pnl"].sum():.2f} ---')

    show_cols = ['entry_px', 'edge', 'exit_px', 'pnl', 'sigma_p']
    print('  All trades:')
    print(tdf[show_cols].describe().round(4).to_string())
    if len(conv) > 0:
        print(f'  Converged exits: median hold = {conv["mins_held"].median():.0f} min, '
              f'mean hold = {conv["mins_held"].mean():.0f} min')
    tdf.to_csv(out_dir / f'backtest_convergence_trades_{pname}.csv', index=False)

TRADING BACKTEST — Convergence Exit  (2025 season, zero explicit fee, spread in BBO)
policy  n_trades  n_converged  n_settlement  mean_edge  hit_rate  mean_pnl  total_pnl    roi  brier  avg_sigma_p  avg_mins_held
    E1       223          164            59     0.1085    0.7399    0.0188     4.2000 0.0530 0.2161       0.0325        36.9000
    E2       182          128            54     0.1238    0.7088    0.0192     3.4900 0.0545 0.2193       0.0331        40.9000
    E3       132           92            40     0.1467    0.6970    0.0306     4.0400 0.0884 0.2305       0.0333        44.1000
    E4        77           51            26     0.1801    0.6623    0.0332     2.5600 0.0991 0.2360       0.0342        47.3000
    S1       180          128            52     0.1245    0.7167    0.0251     4.5200 0.0718 0.2180       0.0329        40.9000
    S2       132           92            40     0.1467    0.6970    0.0306     4.0400 0.0884 0.2305       0.0333        44.1000
    S3       105   

## 7. Hold-to-settlement comparison

Same entry logic (tipoff - 5 min, same policies), but always hold to settlement instead of convergence exit.

In [19]:
# --- Hold-to-settlement backtest ---
def run_backtest_settle(entries_df, policies):
    all_trades = {name: [] for name in policies}

    for _, row in entries_df.iterrows():
        p       = row['p_xgb']
        sigma   = row['sigma_p']
        q_yes   = row['q_yes_exec']
        q_no    = row['q_no_exec']
        hw      = row['home_win']
        game_id = row['game_id']

        edge_yes = p - q_yes
        edge_no  = (1 - p) - q_no

        for pname, pfunc in policies.items():
            if not pfunc(edge_yes, edge_no, sigma):
                continue

            if edge_yes >= edge_no:
                side     = 'YES'
                entry_px = q_yes
                edge     = edge_yes
                payout   = 1.0 if hw == 1 else 0.0
            else:
                side     = 'NO'
                entry_px = q_no
                edge     = edge_no
                payout   = 1.0 if hw == 0 else 0.0

            pnl = payout - entry_px

            all_trades[pname].append({
                'game_id':   game_id,
                'side':      side,
                'entry_px':  entry_px,
                'edge':      edge,
                'pnl':       pnl,
                'payout':    payout,
                'p_xgb':     p,
                'sigma_p':   sigma,
                'home_win':  hw,
            })

    return {k: pd.DataFrame(v) for k, v in all_trades.items()}


trades_settle = run_backtest_settle(entries, POLICIES)

# --- Summary table ---
settle_rows = []
for pname in POLICIES:
    tdf = trades_settle[pname]
    if tdf.empty:
        settle_rows.append({'policy': pname, 'n_trades': 0})
        continue

    n         = len(tdf)
    hit       = (tdf['payout'] == 1.0).mean()
    mean_e    = tdf['edge'].mean()
    mean_pnl  = tdf['pnl'].mean()
    total_pnl = tdf['pnl'].sum()
    total_risk = tdf['entry_px'].sum()
    roi       = total_pnl / total_risk if total_risk > 0 else 0

    brier_p   = np.where(tdf['side'] == 'YES', tdf['p_xgb'], 1 - tdf['p_xgb'])
    brier_y   = tdf['payout'].values
    brier_val = brier_score_loss(brier_y, brier_p)
    avg_sigma = tdf['sigma_p'].mean()

    settle_rows.append({
        'policy':      pname,
        'n_trades':    n,
        'mean_edge':   round(mean_e, 4),
        'hit_rate':    round(hit, 4),
        'mean_pnl':    round(mean_pnl, 4),
        'total_pnl':   round(total_pnl, 2),
        'roi':         round(roi, 4),
        'brier':       round(brier_val, 5),
        'avg_sigma_p': round(avg_sigma, 4),
    })

settle_summary = pd.DataFrame(settle_rows)

print('=' * 120)
print('TRADING BACKTEST — Hold to Settlement  (2025 season, zero explicit fee, spread in BBO)')
print('=' * 120)
print(settle_summary.to_string(index=False))

out_dir = PROJECT_ROOT / 'data' / 'kalshi'
settle_summary.to_csv(out_dir / 'backtest_settlement_summary.csv', index=False)
print(f'\nSaved: {out_dir}/backtest_settlement_summary.csv')

# --- Side-by-side comparison ---
print('\n' + '=' * 120)
print('CONVERGENCE EXIT vs HOLD-TO-SETTLEMENT')
print('=' * 120)
comp_rows = []
for pname in POLICIES:
    c = summary[summary['policy'] == pname].iloc[0]
    s = settle_summary[settle_summary['policy'] == pname].iloc[0]
    comp_rows.append({
        'policy':           pname,
        'n_trades':         int(c['n_trades']),
        'conv_total_pnl':   c['total_pnl'],
        'conv_roi':         c['roi'],
        'conv_hit':         c['hit_rate'],
        'sett_total_pnl':   s['total_pnl'],
        'sett_roi':         s['roi'],
        'sett_hit':         s['hit_rate'],
        'pnl_diff':         round(c['total_pnl'] - s['total_pnl'], 2),
    })
comp_df = pd.DataFrame(comp_rows)
print(comp_df.to_string(index=False))

TRADING BACKTEST — Hold to Settlement  (2025 season, zero explicit fee, spread in BBO)
policy  n_trades  mean_edge  hit_rate  mean_pnl  total_pnl    roi  brier  avg_sigma_p
    E1       223     0.1085    0.3946    0.0394     8.7900 0.1110 0.2161       0.0325
    E2       182     0.1238    0.3846    0.0330     6.0000 0.0938 0.2193       0.0331
    E3       132     0.1467    0.3939    0.0476     6.2800 0.1374 0.2305       0.0333
    E4        77     0.1801    0.4156    0.0801     6.1700 0.2389 0.2360       0.0342
    S1       180     0.1245    0.3889    0.0391     7.0300 0.1116 0.2180       0.0329
    S2       132     0.1467    0.3939    0.0476     6.2800 0.1374 0.2305       0.0333
    S3       105     0.1436    0.3619    0.0110     1.1500 0.0312 0.2238       0.0307
    Z2       155     0.1347    0.3806    0.0416     6.4500 0.1227 0.2187       0.0322

Saved: C:\Users\arius\Desktop\kalshi_wnba_bot\data\kalshi/backtest_settlement_summary.csv

CONVERGENCE EXIT vs HOLD-TO-SETTLEMENT
policy  